# KNN Content-Based Course Recommender

This notebook builds a KNN-based recommender for the Online Course Recommender System project.

The goal is to recommend courses that are similar to a selected course by comparing numerical feature vectors. The notebook follows the processed dataset created in `02_preprocessing.ipynb` and complements the TF-IDF recommender in `03_tfidf_recommender.ipynb`.

## 2. Load Processed Dataset

The processed dataset is loaded from `data/processed/processed_udemy_courses.csv`.

This notebook expects the preprocessing step to have already created the selected recommendation columns, including `combined_features`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

In [ ]:
DATA_PATHS = [
    Path("data/processed/processed_udemy_courses.csv"),
    Path("../data/processed/processed_udemy_courses.csv"),
]

DATA_PATH = next((path for path in DATA_PATHS if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Processed dataset not found. Run 02_preprocessing.ipynb first.")

df = pd.read_csv(DATA_PATH)

print(f"Loaded dataset from: {DATA_PATH}")
print(f"Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns")

The dataset shape confirms how many courses and features are available after preprocessing. The next checks make sure the columns required for KNN are present before model building starts.

In [ ]:
important_columns = [
    "course_id",
    "course_title",
    "subject",
    "level",
    "num_subscribers",
    "num_reviews",
    "price",
    "content_duration",
    "combined_features",
]

df[important_columns].head()

In [ ]:
missing_columns = [column for column in important_columns if column not in df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("All required columns are available for the KNN recommender.")
print(f"Missing combined_features values: {df['combined_features'].isna().sum()}")

## 3. Feature Preparation

KNN compares rows using distances between feature vectors. For this reason, each course needs to be represented with numerical values.

The planned feature set combines text information from `combined_features` with numeric course attributes such as subscribers, reviews, price, and duration.

In [ ]:
text_feature = "combined_features"
numeric_features = [
    "num_subscribers",
    "num_reviews",
    "price",
    "content_duration",
]

df[["course_title", "subject", "level", text_feature] + numeric_features].sample(
    5, random_state=42
)

The text feature gives the main content signal for the recommender. The numeric features add extra course context, such as course popularity, price, and duration.

`subject` and `level` are already included inside `combined_features`, so they do not need to be encoded separately in this first KNN version.

In [ ]:
df[numeric_features].describe()

## 4. Vectorization / Feature Transformation

Text cannot be used directly by KNN. It must first be converted into numerical vectors.

The text features will be transformed with TF-IDF, while numeric columns will be scaled so that large values such as subscribers do not dominate the distance calculation.

In [ ]:
tfidf = TfidfVectorizer(stop_words="english", max_features=1000)
text_matrix = tfidf.fit_transform(df[text_feature].fillna(""))

print(f"Text matrix shape: {text_matrix.shape[0]} courses x {text_matrix.shape[1]} terms")

In [ ]:
scaler = StandardScaler()
numeric_matrix = scaler.fit_transform(df[numeric_features].fillna(0))
numeric_weight = 0.2
weighted_numeric_matrix = numeric_matrix * numeric_weight

print(f"Numeric matrix shape: {numeric_matrix.shape[0]} courses x {numeric_matrix.shape[1]} numeric features")
print(f"Numeric feature weight: {numeric_weight}")

In [ ]:
feature_matrix = np.hstack([text_matrix.toarray(), weighted_numeric_matrix])

print(f"Combined KNN feature matrix shape: {feature_matrix.shape[0]} courses x {feature_matrix.shape[1]} features")

The final feature matrix has one row per course. The first part of the matrix represents course text, while the last columns represent scaled numeric attributes.

The numeric columns are down-weighted so that popularity and price support the recommendations without overpowering the course text.

This keeps the KNN model content-based and avoids using synthetic users or collaborative filtering at this stage.

## 5. Build KNN Model

The KNN model will use scikit-learn to find the nearest courses to a selected course.

The final parameters will be documented after testing a simple working setup.

In [ ]:
knn_model = NearestNeighbors(
    n_neighbors=6,
    metric="cosine",
    algorithm="brute",
)

knn_model.fit(feature_matrix)

print("KNN model fitted successfully.")

The model uses cosine distance because the feature matrix contains many text-based TF-IDF values. A brute-force search is acceptable for this dataset size and keeps the method easy to understand.

`n_neighbors` is set to `6` so that the selected course itself can be removed and five recommendations can still be returned.

## 6. Recommendation Function

The recommendation function will accept a course title and return the top `n` most similar courses.

The function should hide the selected course itself from the output and return a readable table with course metadata and similarity information.

In [ ]:
title_to_index = pd.Series(df.index, index=df["course_title"].str.lower())


def recommend_courses(course_title, n=5):
    """Return the top n courses most similar to the selected course title."""
    normalized_title = course_title.lower()

    if normalized_title not in title_to_index:
        raise ValueError(f"Course title not found: {course_title}")

    course_index = title_to_index[normalized_title]
    if isinstance(course_index, pd.Series):
        course_index = course_index.iloc[0]

    distances, indices = knn_model.kneighbors(feature_matrix[course_index].reshape(1, -1), n_neighbors=n + 1)

    recommendation_rows = []
    for distance, index in zip(distances[0], indices[0]):
        if index == course_index:
            continue

        recommendation_rows.append(
            {
                "course_title": df.loc[index, "course_title"],
                "subject": df.loc[index, "subject"],
                "level": df.loc[index, "level"],
                "price": df.loc[index, "price"],
                "num_subscribers": df.loc[index, "num_subscribers"],
                "distance": round(float(distance), 4),
                "similarity": round(1 - float(distance), 4),
            }
        )

        if len(recommendation_rows) == n:
            break

    return pd.DataFrame(recommendation_rows)

### Recommendation Function Edge Cases

If the selected course title is not found, the function raises an error. If duplicate titles exist, the first matching course is used, which keeps the behavior consistent and simple for this project stage.

## 7. Example Recommendations

The recommender will be tested with examples from different subject areas:

- Python-related course
- Business or finance course
- Web development course

Short observations will be added after the results.

In [ ]:
python_course = "web programming with python"

print(f"Recommendations for: {python_course}")
recommend_courses(python_course, n=5)

The Python example returns mostly web development and programming-related courses. The closest recommendation is another Python web programming course, which shows that the text component is influencing the KNN distance correctly.

Some broader web development courses also appear because they share subject and beginner-level signals with the selected course.

In [ ]:
business_course = "ultimate investment banking course"

print(f"Recommendations for: {business_course}")
recommend_courses(business_course, n=5)

The business example returns courses from the Business Finance subject, especially investment banking and investment-related courses. This is a strong result for a content-based KNN model because the recommendations match both the topic and the subject area.

In [ ]:
web_development_course = "learn complete web development from scratch"

print(f"Recommendations for: {web_development_course}")
recommend_courses(web_development_course, n=5)

The web development example returns courses about web development, JavaScript, HTML/CSS, and learning from scratch. The recommendations are not identical copies of the input course, but they are close enough to be useful alternatives for a learner interested in the same area.

## 8. Strengths and Limitations

This section will summarize what the KNN recommender does well and where it is limited.

## 9. Conclusions

This section will summarize the final KNN pipeline and how it can be compared with the TF-IDF recommender later.